# Finding Patterns Across Series

Time series that *look* similar often share an underlying process -- similar demand
curves, correlated sensor readings, or repeated diurnal patterns. This notebook
explores the full **similarity and clustering** toolkit in
[polars-ts](https://github.com/domenicocinque/polars-ts):

1. **Distance metrics** -- 12 pairwise distance functions covering elastic (DTW family),
   edit-based (ERP, EDR, LCSS), and shape-based (SBD, Frechet) paradigms.
2. **Partitional clustering** -- K-Medoids (distance-matrix based) and K-Shape (shape-based).
3. **Density-based clustering** -- HDBSCAN (automatic k, noise detection) and DBSCAN
   (fixed-radius neighbourhood).
4. **Spectral clustering** -- graph Laplacian eigendecomposition with Gaussian kernel
   affinity (K-Spectral Centroid).
5. **Cluster evaluation** -- Silhouette, Davies-Bouldin, and Calinski-Harabasz scores.
6. **Classification** -- KNN and K-Shape classifiers that leverage the same distance
   infrastructure.
7. **Multivariate distances** -- extensions of DTW and MSM for multi-column series.

**Dataset**: M4 hourly competition data (first 20 series), which naturally contain
diverse patterns -- some trending, some seasonal, some flat -- making them ideal for
clustering experiments.

### References

- Paparrizos, J. & Gravano, L. (2015). *k-Shape: Efficient and Accurate Clustering of Time Series*. SIGMOD.
- Campello, R.J.G.B. et al. (2013). *Density-Based Clustering Based on Hierarchical Density Estimates*. PAKDD.
- von Luxburg, U. (2007). *A Tutorial on Spectral Clustering*. Statistics and Computing.
- Petitjean, F. et al. (2011). *A global averaging method for dynamic time warping*. Pattern Recognition.
- Loning, M. et al. (2024). [*Modern Time Series Forecasting with Python*, 2nd ed.](https://www.packtpub.com/product/modern-time-series-forecasting-with-python-second-edition/9781803246802)

In [ ]:
import hvplot.polars  # noqa
import numpy as np
import polars as pl

from polars_ts import (
    compute_pairwise_dtw,
    compute_pairwise_ddtw,
    compute_pairwise_wdtw,
    compute_pairwise_msm,
    compute_pairwise_erp,
    compute_pairwise_lcss,
    compute_pairwise_twe,
    compute_pairwise_sbd,
    compute_pairwise_frechet,
    compute_pairwise_edr,
    compute_pairwise_dtw_multi,
    compute_pairwise_msm_multi,
    compute_pairwise_distance,
    kmedoids,
    KShape,
    hdbscan_cluster,
    dbscan_cluster,
    spectral_cluster,
    silhouette_score,
    silhouette_samples,
    davies_bouldin_score,
    calinski_harabasz_score,
    knn_classify,
    TimeSeriesKNNClassifier,
    KShapeClassifier,
)

## 1. Load and normalize the data

We take the first 20 series from the M4 hourly dataset. Before computing distances we
**z-normalize** each series so that differences in scale do not dominate the results --
this is standard practice for shape-based comparisons.

In [ ]:
df = (
    pl.scan_parquet("https://datasets-nixtla.s3.amazonaws.com/m4-hourly.parquet")
    .filter(pl.col("unique_id").str.strip_chars_start("H").cast(pl.Int64) <= 20)
    .collect()
)

# Z-normalize per series
df = df.with_columns(((pl.col("y") - pl.mean("y").over("unique_id")) / pl.std("y").over("unique_id")).alias("y"))

print(f"Shape: {df.shape}")
print(f"Series: {df['unique_id'].n_unique()}")
df.head(10)

In [ ]:
df.hvplot.line(
    x="ds",
    y="y",
    by="unique_id",
    width=900,
    height=400,
    title="M4 Hourly -- 20 Normalized Series",
    alpha=0.7,
)

## 2. Elastic distance metrics: DTW, DDTW, WDTW

Dynamic Time Warping (DTW) aligns two series by warping the time axis to minimize
the total alignment cost. Variants include:

- **DTW** -- classic unconstrained or band-constrained (Sakoe-Chiba) warping.
- **DDTW** -- derivative DTW; aligns the first differences rather than raw values,
  reducing the sensitivity to y-offset.
- **WDTW** -- weighted DTW; applies a logistic weight that penalizes large warp steps.

All `compute_pairwise_*` functions return a tidy DataFrame with columns
`[id_1, id_2, <metric>]`.

In [ ]:
# Unconstrained DTW
dtw_df = compute_pairwise_dtw(df, df)
print("DTW distances (head):")
dtw_df.head(10)

In [ ]:
# Sakoe-Chiba constrained DTW (band width = 10)
dtw_sc_df = compute_pairwise_dtw(df, df, method="sakoe_chiba", param=10.0)
print("Constrained DTW (Sakoe-Chiba, r=10):")
dtw_sc_df.head(5)

In [ ]:
# Derivative DTW
ddtw_df = compute_pairwise_ddtw(df, df)
print("DDTW distances (head):")
ddtw_df.head(5)

In [ ]:
# Weighted DTW with penalty parameter g=0.05
wdtw_df = compute_pairwise_wdtw(df, df, g=0.05)
print("WDTW distances (g=0.05):")
wdtw_df.head(5)

## 3. More distance metrics: MSM, ERP, LCSS, TWE, SBD, Frechet, EDR

Beyond elastic warping, polars-ts provides edit-based and shape-based distances:

| Metric | Type | Key parameter | Notes |
|--------|------|---------------|-------|
| **MSM** | Edit | `c` (move cost) | Move-Split-Merge; robust to phase shifts |
| **ERP** | Edit | `g` (gap value) | Edit distance with Real Penalty |
| **LCSS** | Similarity | `epsilon` (match threshold) | Longest Common Subsequence |
| **TWE** | Elastic | `nu`, `lambda_` | Time Warp Edit; penalizes stiffness |
| **SBD** | Shape | -- | Shape-Based Distance via cross-correlation |
| **Frechet** | Geometric | -- | Maximum detour on the best coupling |
| **EDR** | Edit | -- | Edit Distance on Real sequences |

In [ ]:
msm_df = compute_pairwise_msm(df, df, c=1.0)
erp_df = compute_pairwise_erp(df, df, g=0.0)
lcss_df = compute_pairwise_lcss(df, df, epsilon=0.5)
twe_df = compute_pairwise_twe(df, df, 0.01, 1.0)
sbd_df = compute_pairwise_sbd(df, df)
frechet_df = compute_pairwise_frechet(df, df)
edr_df = compute_pairwise_edr(df, df)

for name, d in [
    ("MSM", msm_df),
    ("ERP", erp_df),
    ("LCSS", lcss_df),
    ("TWE", twe_df),
    ("SBD", sbd_df),
    ("Frechet", frechet_df),
    ("EDR", edr_df),
]:
    print(f"{name:8s} -- shape: {d.shape}, columns: {d.columns}")

## 4. Unified distance API: `compute_pairwise_distance`

Rather than calling each function individually, `compute_pairwise_distance` accepts a
`method` string. This is especially handy when you want to loop over metrics or
parameterize from a config file.

In [ ]:
# Unified API -- same result as compute_pairwise_dtw(df, df)
dist_unified = compute_pairwise_distance(df, df, method="dtw")
print("Unified API result:")
dist_unified.head(5)

In [ ]:
# Visualize DTW distance matrix as a heatmap
dtw_df.hvplot.heatmap(
    x="id_1",
    y="id_2",
    C="dtw",
    width=600,
    height=500,
    cmap="viridis",
    title="Pairwise DTW Distance Matrix",
    rot=45,
)

## 5. K-Medoids clustering

K-Medoids selects *actual series* as cluster centres (medoids), making it
interpretable. It works directly with any pairwise distance matrix, so we can swap
the underlying metric easily.

We cluster the 20 series into 3 groups using DTW.

In [ ]:
# Functional API
clusters_kmed = kmedoids(df, k=3, method="dtw")
print("K-Medoids cluster assignments:")
clusters_kmed

In [ ]:
# Visualize: overlay cluster labels on the raw series
df_clustered = df.join(clusters_kmed, on="unique_id")

df_clustered.hvplot.line(
    x="ds",
    y="y",
    by="unique_id",
    groupby="cluster",
    width=900,
    height=400,
    title="Series Colored by K-Medoids Cluster (k=3, DTW)",
    alpha=0.7,
)

## 6. K-Shape clustering

K-Shape uses the shape-based distance (SBD) and a specialized centroid computation
based on cross-correlation. It tends to group series that share the same *shape*
regardless of amplitude or phase.

In [ ]:
kshape = KShape(n_clusters=3)
kshape.fit(df)

print(f"Labels (first 10): {kshape.labels_[:10]}")
print(f"Number of centroids: {len(kshape.centroids_)}")

In [ ]:
# Build a cluster-assignment DataFrame from KShape labels
series_ids = df.select("unique_id").unique(maintain_order=True)["unique_id"].to_list()
kshape_labels = pl.DataFrame(
    {
        "unique_id": series_ids,
        "cluster": kshape.labels_,
    }
)

df_kshape = df.join(kshape_labels, on="unique_id")

df_kshape.hvplot.line(
    x="ds",
    y="y",
    by="unique_id",
    groupby="cluster",
    width=900,
    height=400,
    title="Series Colored by K-Shape Cluster (k=3)",
    alpha=0.7,
)

## 7. HDBSCAN -- density-based clustering

HDBSCAN (Hierarchical DBSCAN) discovers clusters of varying density without
requiring you to specify the number of clusters upfront. Series that do not
belong to any dense region are labelled as **noise** (`cluster = -1`).

The main tuning knob is `min_cluster_size` -- the smallest group of series
that should be considered a cluster.

In [ ]:
# HDBSCAN with DTW distances
clusters_hdbscan = hdbscan_cluster(df, method="dtw", min_cluster_size=3)
print("HDBSCAN cluster assignments:")
print(clusters_hdbscan)

n_clusters = clusters_hdbscan.filter(pl.col("cluster") >= 0)["cluster"].n_unique()
n_noise = clusters_hdbscan.filter(pl.col("cluster") == -1).height
print(f"\nClusters found: {n_clusters}, Noise points: {n_noise}")

In [ ]:
# Tune min_cluster_size to see how it affects the number of clusters
for mcs in [2, 3, 5, 7]:
    res = hdbscan_cluster(df, method="dtw", min_cluster_size=mcs)
    nc = res.filter(pl.col("cluster") >= 0)["cluster"].n_unique()
    nn = res.filter(pl.col("cluster") == -1).height
    print(f"min_cluster_size={mcs}: {nc} clusters, {nn} noise")

In [ ]:
# Visualize HDBSCAN results -- noise series shown separately
df_hdbscan = df.join(clusters_hdbscan, on="unique_id").with_columns(
    pl.when(pl.col("cluster") == -1)
    .then(pl.lit("noise"))
    .otherwise(pl.col("cluster").cast(pl.Utf8))
    .alias("cluster_label")
)

df_hdbscan.hvplot.line(
    x="ds",
    y="y",
    by="unique_id",
    groupby="cluster_label",
    width=900,
    height=400,
    title="HDBSCAN Clusters (DTW, min_cluster_size=3)",
    alpha=0.7,
)

## 8. DBSCAN -- eps sensitivity

DBSCAN groups series whose pairwise distance is below a threshold `eps`.
Unlike HDBSCAN it uses a **fixed** density threshold, so the choice of `eps`
is critical -- too small and everything is noise, too large and everything
merges into one cluster.

In [ ]:
# Sweep eps to see its effect on cluster count and noise
for eps in [0.5, 1.0, 2.0, 5.0, 10.0]:
    res = dbscan_cluster(df, method="dtw", eps=eps, min_samples=2)
    nc = res.filter(pl.col("cluster") >= 0)["cluster"].n_unique()
    nn = res.filter(pl.col("cluster") == -1).height
    print(f"eps={eps:5.1f}: {nc} clusters, {nn} noise")

In [ ]:
# Pick a reasonable eps and visualize
clusters_dbscan = dbscan_cluster(df, method="dtw", eps=5.0, min_samples=2)
print("DBSCAN cluster assignments (eps=5.0):")
print(clusters_dbscan)

df_dbscan = df.join(clusters_dbscan, on="unique_id").with_columns(
    pl.when(pl.col("cluster") == -1)
    .then(pl.lit("noise"))
    .otherwise(pl.col("cluster").cast(pl.Utf8))
    .alias("cluster_label")
)

df_dbscan.hvplot.line(
    x="ds",
    y="y",
    by="unique_id",
    groupby="cluster_label",
    width=900,
    height=400,
    title="DBSCAN Clusters (DTW, eps=5.0, min_samples=2)",
    alpha=0.7,
)

## 9. Spectral clustering (KSC)

Spectral clustering captures **global graph structure** that local-search methods
like K-Medoids can miss. It works by:

1. Computing pairwise distances and converting them into a Gaussian kernel affinity
   matrix: `A[i,j] = exp(-d² / 2σ²)`.
2. Building the normalized graph Laplacian.
3. Extracting the `k` smallest eigenvectors.
4. Running k-means on the resulting spectral embedding.

The bandwidth parameter `sigma` controls how quickly affinity decays with distance --
small sigma makes the graph sparser and can split marginal connections.

In [ ]:
# Spectral clustering with SBD kernel
clusters_spectral_sbd = spectral_cluster(df, k=3, method="sbd", sigma=1.0)
print("Spectral clustering (SBD, k=3, sigma=1.0):")
print(clusters_spectral_sbd)

# Also try with DTW kernel
clusters_spectral_dtw = spectral_cluster(df, k=3, method="dtw", sigma=1.0)
print("\nSpectral clustering (DTW, k=3, sigma=1.0):")
print(clusters_spectral_dtw)

In [ ]:
# Sigma sensitivity -- how bandwidth affects cluster boundaries
for sigma in [0.1, 0.5, 1.0, 2.0, 5.0]:
    res = spectral_cluster(df, k=3, method="sbd", sigma=sigma)
    label_counts = res.group_by("cluster").len().sort("cluster")
    sizes = label_counts["len"].to_list()
    print(f"sigma={sigma:4.1f}: cluster sizes = {sizes}")

In [ ]:
# Visualize spectral clustering result
df_spectral = df.join(clusters_spectral_sbd, on="unique_id")

df_spectral.hvplot.line(
    x="ds",
    y="y",
    by="unique_id",
    groupby="cluster",
    width=900,
    height=400,
    title="Spectral Clusters (SBD, k=3, sigma=1.0)",
    alpha=0.7,
)

In [ ]:
# Spectral embedding visualization -- scatter the first 2 Laplacian eigenvectors
from scipy.linalg import eigh

# Compute SBD distance matrix and build the Laplacian
sbd_mat = compute_pairwise_sbd(df, df)
series_ids = df.select("unique_id").unique(maintain_order=True)["unique_id"].to_list()
n = len(series_ids)
id_to_idx = {s: i for i, s in enumerate(series_ids)}

D = np.zeros((n, n))
for row in sbd_mat.iter_rows(named=True):
    i, j = id_to_idx[row["id_1"]], id_to_idx[row["id_2"]]
    D[i, j] = D[j, i] = row["sbd"]

sigma = 1.0
A = np.exp(-D**2 / (2 * sigma**2))
np.fill_diagonal(A, 0)
D_diag = np.diag(A.sum(axis=1) ** -0.5)
L_sym = np.eye(n) - D_diag @ A @ D_diag

eigvals, eigvecs = eigh(L_sym, subset_by_index=[0, 2])
embedding = eigvecs[:, 1:]  # skip trivial eigenvector

labels = clusters_spectral_sbd["cluster"].to_list()
embed_df = pl.DataFrame({
    "eigvec_1": embedding[:, 0],
    "eigvec_2": embedding[:, 1],
    "cluster": [str(l) for l in labels],
    "unique_id": series_ids,
})

embed_df.hvplot.scatter(
    x="eigvec_1",
    y="eigvec_2",
    by="cluster",
    hover_cols=["unique_id"],
    width=600,
    height=400,
    title="Spectral Embedding (first 2 Laplacian eigenvectors)",
    size=100,
)

## 10. Cluster evaluation

How good are our clusters? Three complementary metrics help us decide:

| Metric | Ideal value | Interpretation |
|--------|-------------|----------------|
| **Silhouette** | Close to 1 | Cohesive within, separated between clusters |
| **Davies-Bouldin** | Close to 0 | Low intra-cluster scatter relative to inter-cluster distance |
| **Calinski-Harabasz** | Higher is better | Ratio of between-cluster to within-cluster dispersion |

In [ ]:
kmed_labels = clusters_kmed["cluster"].to_list()

sil = silhouette_score(df, kmed_labels, method="dtw")
dbi = davies_bouldin_score(df, kmed_labels, method="dtw")
chi = calinski_harabasz_score(df, kmed_labels, method="dtw")

print("K-Medoids (DTW, k=3)")
print(f"  Silhouette score:       {sil:.4f}")
print(f"  Davies-Bouldin index:   {dbi:.4f}")
print(f"  Calinski-Harabasz index: {chi:.4f}")

# Per-sample silhouette scores for deeper analysis
sil_samples = silhouette_samples(df, kmed_labels, method="dtw")
print("\nPer-series silhouette scores:")
sil_samples

In [ ]:
# Compare evaluation across different k values
results = []
for k in [2, 3, 4, 5]:
    cl = kmedoids(df, k=k, method="dtw")
    labels = cl["cluster"].to_list()
    results.append(
        {
            "k": k,
            "silhouette": silhouette_score(df, labels, method="dtw"),
            "davies_bouldin": davies_bouldin_score(df, labels, method="dtw"),
            "calinski_harabasz": calinski_harabasz_score(df, labels, method="dtw"),
        }
    )

eval_df = pl.DataFrame(results)
eval_df

### Comparison across clustering methods

Evaluate HDBSCAN, DBSCAN, and Spectral alongside K-Medoids and K-Shape.
For density-based methods we exclude noise points (`cluster = -1`) from
evaluation since silhouette is undefined for singleton noise labels.

In [ ]:
# Build comparison table
comparison = []

# K-Medoids
kmed_labels = clusters_kmed["cluster"].to_list()
comparison.append({
    "method": "K-Medoids (DTW)",
    "n_clusters": len(set(kmed_labels)),
    "noise": 0,
    "silhouette": silhouette_score(df, kmed_labels, method="dtw"),
    "davies_bouldin": davies_bouldin_score(df, kmed_labels, method="dtw"),
    "calinski_harabasz": calinski_harabasz_score(df, kmed_labels, method="dtw"),
})

# K-Shape
kshape_labs = kshape.labels_
comparison.append({
    "method": "K-Shape (SBD)",
    "n_clusters": len(set(kshape_labs)),
    "noise": 0,
    "silhouette": silhouette_score(df, kshape_labs, method="sbd"),
    "davies_bouldin": davies_bouldin_score(df, kshape_labs, method="sbd"),
    "calinski_harabasz": calinski_harabasz_score(df, kshape_labs, method="sbd"),
})

# Spectral (SBD)
spec_labels = clusters_spectral_sbd["cluster"].to_list()
comparison.append({
    "method": "Spectral (SBD)",
    "n_clusters": len(set(spec_labels)),
    "noise": 0,
    "silhouette": silhouette_score(df, spec_labels, method="sbd"),
    "davies_bouldin": davies_bouldin_score(df, spec_labels, method="sbd"),
    "calinski_harabasz": calinski_harabasz_score(df, spec_labels, method="sbd"),
})

# HDBSCAN -- evaluate only non-noise series
hdb_all = clusters_hdbscan["cluster"].to_list()
hdb_non_noise = clusters_hdbscan.filter(pl.col("cluster") >= 0)
n_noise_hdb = sum(1 for l in hdb_all if l == -1)
if hdb_non_noise["cluster"].n_unique() >= 2:
    hdb_ids = hdb_non_noise["unique_id"].to_list()
    df_hdb_clean = df.filter(pl.col("unique_id").is_in(hdb_ids))
    hdb_clean_labels = hdb_non_noise["cluster"].to_list()
    comparison.append({
        "method": "HDBSCAN (DTW)",
        "n_clusters": hdb_non_noise["cluster"].n_unique(),
        "noise": n_noise_hdb,
        "silhouette": silhouette_score(df_hdb_clean, hdb_clean_labels, method="dtw"),
        "davies_bouldin": davies_bouldin_score(df_hdb_clean, hdb_clean_labels, method="dtw"),
        "calinski_harabasz": calinski_harabasz_score(df_hdb_clean, hdb_clean_labels, method="dtw"),
    })

# DBSCAN -- evaluate only non-noise series
dbs_all = clusters_dbscan["cluster"].to_list()
dbs_non_noise = clusters_dbscan.filter(pl.col("cluster") >= 0)
n_noise_dbs = sum(1 for l in dbs_all if l == -1)
if dbs_non_noise["cluster"].n_unique() >= 2:
    dbs_ids = dbs_non_noise["unique_id"].to_list()
    df_dbs_clean = df.filter(pl.col("unique_id").is_in(dbs_ids))
    dbs_clean_labels = dbs_non_noise["cluster"].to_list()
    comparison.append({
        "method": "DBSCAN (DTW)",
        "n_clusters": dbs_non_noise["cluster"].n_unique(),
        "noise": n_noise_dbs,
        "silhouette": silhouette_score(df_dbs_clean, dbs_clean_labels, method="dtw"),
        "davies_bouldin": davies_bouldin_score(df_dbs_clean, dbs_clean_labels, method="dtw"),
        "calinski_harabasz": calinski_harabasz_score(df_dbs_clean, dbs_clean_labels, method="dtw"),
    })

comp_df = pl.DataFrame(comparison)
print("Clustering method comparison:")
comp_df

## 11. KNN classification

Once we have cluster labels, we can treat them as ground-truth classes and build a
**k-nearest-neighbours classifier**. We split the labelled data into 70/30
train/test, then predict the held-out labels.

In [ ]:
# Use K-Medoids labels as ground truth
labelled = df.join(
    clusters_kmed.rename({"cluster": "label"}),
    on="unique_id",
)

# 70/30 split by unique_id
all_ids = labelled.select("unique_id").unique(maintain_order=True)["unique_id"].to_list()
rng = np.random.default_rng(42)
rng.shuffle(all_ids)
split = int(0.7 * len(all_ids))
train_ids, test_ids = all_ids[:split], all_ids[split:]

train_df = labelled.filter(pl.col("unique_id").is_in(train_ids))
test_df = labelled.filter(pl.col("unique_id").is_in(test_ids))

print(f"Train series: {len(train_ids)}, Test series: {len(test_ids)}")

In [ ]:
# Functional API
preds_fn = knn_classify(train_df, test_df, k=3, method="dtw", label_col="label")
print("KNN predictions (functional API):")
preds_fn

In [ ]:
# OOP API -- TimeSeriesKNNClassifier
knn = TimeSeriesKNNClassifier(k=3, metric="dtw")
knn.fit(train_df, label_col="label")
preds_oop = knn.predict(test_df)
print("KNN predictions (OOP API):")
preds_oop

## 12. K-Shape classifier

The `KShapeClassifier` learns centroids *per class* during training, then assigns
test series to the class whose centroid is closest in SBD. This is often faster than
KNN because prediction only requires comparing against a handful of centroids rather
than the entire training set.

In [ ]:
kshape_clf = KShapeClassifier(n_centroids_per_class=1)
kshape_clf.fit(train_df, label_col="label")
preds_kshape = kshape_clf.predict(test_df)
print("K-Shape classifier predictions:")
preds_kshape

## 13. Multivariate distances

When series have more than one value column (e.g., temperature *and* humidity), we
need multivariate distance functions. polars-ts provides `compute_pairwise_dtw_multi`
and `compute_pairwise_msm_multi` for this purpose.

We simulate a second channel by adding a lagged copy of `y` as `y2`.

In [ ]:
# Create a second value column (lag-1 of y) to demonstrate multivariate distances
df_multi = df.with_columns(pl.col("y").shift(1).over("unique_id").fill_null(0.0).alias("y2"))
print(f"Multivariate shape: {df_multi.shape}")
df_multi.head(5)

In [ ]:
# Multivariate DTW (Euclidean point-wise distance across channels)
dtw_multi = compute_pairwise_dtw_multi(df_multi, df_multi, metric="euclidean")
print("Multivariate DTW:")
dtw_multi.head(5)

In [ ]:
# Multivariate MSM
msm_multi = compute_pairwise_msm_multi(df_multi, df_multi, c=1.0)
print("Multivariate MSM:")
msm_multi.head(5)

## Summary

This notebook covered the complete similarity, clustering, and classification
pipeline in polars-ts:

| Category | APIs | Purpose |
|----------|------|---------|
| Elastic distances | `compute_pairwise_dtw`, `ddtw`, `wdtw` | Warp-based alignment |
| Edit distances | `compute_pairwise_msm`, `erp`, `lcss`, `twe`, `edr` | Edit-operation costs |
| Shape distances | `compute_pairwise_sbd`, `frechet` | Cross-correlation / geometric |
| Unified API | `compute_pairwise_distance` | Single entry point, method string |
| Partitional clustering | `kmedoids`, `KShape` | Fixed-k centroid-based grouping |
| Density clustering | `hdbscan_cluster`, `dbscan_cluster` | Automatic k, noise detection |
| Spectral clustering | `spectral_cluster` | Graph Laplacian eigenvector grouping |
| Evaluation | `silhouette_score`, `davies_bouldin_score`, `calinski_harabasz_score` | Measure cluster quality |
| Classification | `knn_classify`, `TimeSeriesKNNClassifier`, `KShapeClassifier` | Predict labels |
| Multivariate | `compute_pairwise_dtw_multi`, `compute_pairwise_msm_multi` | Multi-channel series |

### When to use which clustering method

- **K-Medoids / K-Shape**: when you know `k` in advance and want interpretable
  medoids or shape-based centroids.
- **HDBSCAN**: when you don't know `k` and want to discover clusters of varying
  density while automatically detecting outliers.
- **DBSCAN**: when you have a clear distance threshold (`eps`) that defines
  "similar enough", e.g., from domain knowledge.
- **Spectral clustering**: when cluster structure is non-convex or graph-like --
  spectral methods capture global connectivity that local-search methods can miss.

### Tips for real-world use

- **Normalize first.** Z-normalization ensures distances reflect shape, not scale.
- **Start with DTW or SBD.** They are well-understood defaults; switch to MSM or ERP
  if you need edit-distance semantics.
- **Use the silhouette score** to pick `k` -- it balances cohesion and separation.
- **K-Shape is fast** because SBD leverages FFT-based cross-correlation.
- **HDBSCAN handles noise** -- series that don't fit any cluster are labelled `-1`.
- **Spectral sigma matters** -- too small fragments clusters, too large merges them.

### Further reading

- Paparrizos, J. & Gravano, L. (2015). *k-Shape: Efficient and Accurate Clustering of Time Series*. SIGMOD.
- Berndt, D.J. & Clifford, J. (1994). *Using Dynamic Time Warping to Find Patterns in Time Series*. KDD Workshop.
- Stefan, A. et al. (2013). *The Move-Split-Merge Metric for Time Series*. IEEE TKDE.
- Campello, R.J.G.B. et al. (2013). *Density-Based Clustering Based on Hierarchical Density Estimates*. PAKDD.
- von Luxburg, U. (2007). *A Tutorial on Spectral Clustering*. Statistics and Computing.
- Polars documentation: <https://docs.pola.rs/>
- hvPlot documentation: <https://hvplot.holoviz.org/>
- polars-ts repository: <https://github.com/domenicocinque/polars-ts>